# MCRDR rerun of the VAREX skill-guidance pilot

This notebook repeats the pilot deck's three paired runs with the **MCRDR** engine serving the reduced skill:
Qwen3.7 Flash through OpenRouter (provider `alibaba`), temperature 0, 8,192 max tokens, the same 30 Flat dev
documents, rolling skill loading, and 20 model calls / 10 minutes per episode. Each run is **full + reduced**,
so 60 episodes per run and **180 episodes** in total. The skill-blind arm never touches the engine, so the
partner's skill-blind runs are reused for the comparison.

**Cost and time.** These are paid model calls: roughly $0.02 per episode at the configured price ceilings,
so about **$3.70** for all three runs, and several hours at one episode at a time.

**How to run it**

1. In VS Code, install the **Google Colab** extension, open this notebook, choose **Select Kernel → Colab**
   and start a server (a CPU runtime is enough). The Colab web UI works too.
2. **Run All.** The first cell asks for your OpenRouter key in a hidden prompt; it is kept only in the
   runtime's memory and never written to a file.
3. Keep the runtime connected. If it drops, reconnect and run the cells again: finished episodes are kept
   and each run resumes where it stopped.

Results stay on the runtime under `/content/rdr-skills-mcrdr/experiments/`; the last cell downloads a zip.
Episode records contain document text, so keep them private.

In [ ]:
# Settings. The defaults repeat the pilot deck's runs; change them only deliberately.
BRANCH = "MCRDR-testing"
WORKDIR = "/content/rdr-skills-mcrdr"
EXPERIMENT = "qwen-dev30"  # examples/agent/varex.mcrdr.qwen-dev30.json
RUN_NAME = "mcrdr-qwen-rolling-dev30-full-reduced-run{}"

# Optional: your partner's single-classification (SCRDR) run folders, for the side-by-side report.
# Zip their run folders, then either upload the zip to SCRDR_ZIP or put a download link in SCRDR_ZIP_URL.
SCRDR_ZIP = "/content/scrdr-results.zip"
SCRDR_ZIP_URL = ""
# Folder names inside the zip; edit them to match your partner's run names.
SCRDR_RUNS = {
    f"Run {n}": [f"qwen-rolling-dev30-full-reduced-run{n}", f"qwen-skill-blind-dev30-run{n}"]
    for n in (1, 2, 3)
}

## 1. Model key

In [ ]:
import getpass
import os

if not os.environ.get("MODEL_API_KEY"):
    key = None
    try:  # Colab Secrets, if this notebook has been granted access to a MODEL_API_KEY secret
        from google.colab import userdata
        key = userdata.get("MODEL_API_KEY")
    except Exception:
        pass
    os.environ["MODEL_API_KEY"] = key or getpass.getpass("OpenRouter key (hidden, not saved): ")
print("MODEL_API_KEY is set" if os.environ.get("MODEL_API_KEY") else "MODEL_API_KEY is missing")

## 2. Setup script

Fetches `scripts/setup_mcrdr_colab.py` from the branch. It needs only the standard library.

In [ ]:
import importlib.util
import sys
import urllib.request
from pathlib import Path

SCRIPT = Path("/content/setup_mcrdr_colab.py")
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/dmw2101/RDR-skills-/{BRANCH}/scripts/setup_mcrdr_colab.py", SCRIPT)
spec = importlib.util.spec_from_file_location("setup_mcrdr_colab", SCRIPT)
setup_mcrdr_colab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup_mcrdr_colab
spec.loader.exec_module(setup_mcrdr_colab)
EXPERIMENTS_DIR = Path(WORKDIR) / "experiments"


def mcrdr_run(number, *, execute=True):
    """Set up (idempotently) and, with execute=True, run, inspect and summarize one full+reduced run."""
    name = RUN_NAME.format(number)
    args = ["--experiment", EXPERIMENT, "--conditions", "full", "reduced", "--run-name", name,
            "--workdir", WORKDIR, "--branch", BRANCH]
    if (EXPERIMENTS_DIR / "results" / name).exists():
        args.append("--resume")  # keep finished episodes; never start a second copy of a run
    if execute:
        args.append("--run")
    if setup_mcrdr_colab.main(args):
        raise RuntimeError(f"{name} stopped; read the message above, fix it, and rerun this cell")
    print(f"\n{name}: {'finished and summarized' if execute else 'set up and checked'}")

## 3. Set up and check, without model calls

Installs the PDF tools and Python packages, clones the branch, prepares the 30 documents, freezes the PDF
skill snapshot, starts the MCRDR skill service (port 8766), and runs `doctor`. Nothing is spent here.

In [ ]:
mcrdr_run(1, execute=False)

## 4. Run 1 (paid: 60 episodes)

In [ ]:
mcrdr_run(1)

## 5. Run 2 (paid: 60 episodes)

In [ ]:
mcrdr_run(2)

## 6. Run 3 (paid: 60 episodes)

In [ ]:
mcrdr_run(3)

## 7. Optional: your partner's SCRDR run folders

With them, the report adds the skill-blind column (reused from their runs) and a direct SCRDR-vs-MCRDR
comparison. Without them, the report covers the MCRDR full and reduced arms only, and step 9 still compares
those with the deck's published numbers.

In [ ]:
import zipfile

SCRDR_DIR = EXPERIMENTS_DIR / "scrdr"
if SCRDR_ZIP_URL and not Path(SCRDR_ZIP).exists():
    urllib.request.urlretrieve(SCRDR_ZIP_URL, SCRDR_ZIP)
if Path(SCRDR_ZIP).exists():
    with zipfile.ZipFile(SCRDR_ZIP) as archive:
        archive.extractall(SCRDR_DIR)


def locate(name):
    """Path of a run folder (relative to experiments/), searching the extracted SCRDR folders by name."""
    for folder in sorted(SCRDR_DIR.rglob("manifest.json")) if SCRDR_DIR.exists() else []:
        if folder.parent.name == name:
            return folder.parent.relative_to(EXPERIMENTS_DIR).as_posix()
    return None


found = sorted(p.parent.relative_to(EXPERIMENTS_DIR).as_posix() for p in SCRDR_DIR.rglob("manifest.json")) \
    if SCRDR_DIR.exists() else []
print("SCRDR run folders found:", *found or ["none"], sep="\n  ")

## 8. Report

In [ ]:
import json
import subprocess

from IPython.display import Markdown, display

series = {"MCRDR": {}}
for number in (1, 2, 3):
    folders = [f"results/{RUN_NAME.format(number)}"]
    if not (EXPERIMENTS_DIR / folders[0] / "summary.json").exists():
        print(f"Run {number} is not finished and summarized yet; leaving it out")
        continue
    blind = locate(SCRDR_RUNS[f"Run {number}"][1])
    series["MCRDR"][f"Run {number}"] = folders + ([blind] if blind else [])
scrdr = {run: [locate(name) for name in names] for run, names in SCRDR_RUNS.items()}
scrdr = {run: folders for run, folders in scrdr.items() if all(folders)}
if scrdr:
    series = {"SCRDR": scrdr, **series}
if not series["MCRDR"]:
    raise RuntimeError("no finished MCRDR runs to report yet")
SPEC = EXPERIMENTS_DIR / "report-spec.json"
SPEC.write_text(json.dumps({"series": series}, indent=2), encoding="utf-8")
python = Path(WORKDIR) / ".venv" / "bin" / "python"
result = subprocess.run([str(python), "-m", "skill_agent", "report", "--spec", str(SPEC)], cwd=WORKDIR,
                        capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr or result.stdout)
display(Markdown((EXPERIMENTS_DIR / "report" / "report.md").read_text(encoding="utf-8")))

## 9. MCRDR next to the deck's published SCRDR numbers

In [ ]:
# Published in VAREX-skill-guidance-pilot-v2 (single-classification engine, 3 runs x 30 documents).
DECK = {
    "field_exact_match": {
        "Run 1": {"full": 84.15, "reduced": 91.70, "none": 86.79},
        "Run 2": {"full": 87.17, "reduced": 85.66, "none": 79.25},
        "Run 3": {"full": 95.85, "reduced": 93.96, "none": 82.64},
        "Pooled": {"full": 89.06, "reduced": 90.44, "none": 82.89},
    },
    "fields_correct": {"full": 708, "reduced": 719, "none": 659},
    "fields_total": 795,
    "episodes": 90,
    "completed": {"full": 88, "reduced": 87, "none": 78},
    "perfect_documents": {"full": 62, "reduced": 68, "none": 61},
    "median_input_tokens": {"full": 51094, "reduced": 43238, "none": 18828},
    "mean_input_tokens": {"full": 76900, "reduced": 77400, "none": 49800},
    "reduced_minus_full_pp": {"Run 1": 7.55, "Run 2": -1.51, "Run 3": -1.89, "Pooled": 1.38},
    "top5_input_share_pct": {"full": 37, "reduced": 42, "none": 43},
}

In [ ]:
report = json.loads((EXPERIMENTS_DIR / "report" / "report.json").read_text(encoding="utf-8"))
mcrdr = report["series"]["MCRDR"]
pooled = mcrdr["pooled"]["conditions"]


def cell(value, kind):
    if value is None:
        return "—"
    return {"pct": f"{value:.2f}%", "pp": f"{value:+.2f} pp", "int": f"{round(value):,}",
            "k": f"{value / 1000:.1f}k"}[kind]


rows = []
for run in ("Run 1", "Run 2", "Run 3"):
    stats = mcrdr["runs"].get(run, {}).get("conditions", {})
    for condition in ("full", "reduced"):
        value = stats.get(condition, {}).get("field_exact_match")
        rows.append((f"Exact match {run}, {condition}", cell(DECK["field_exact_match"][run][condition], "pct"),
                     cell(None if value is None else value * 100, "pct")))
for condition in ("full", "reduced"):
    stats = pooled.get(condition, {})
    rows += [
        (f"Exact match pooled, {condition}", cell(DECK["field_exact_match"]["Pooled"][condition], "pct"),
         cell(stats.get("field_exact_match", 0) * 100 if stats else None, "pct")),
        (f"Correct fields, {condition}", f"{DECK['fields_correct'][condition]} / {DECK['fields_total']}",
         f"{stats.get('fields_correct')} / {stats.get('fields_total')}" if stats else "—"),
        (f"Completed, {condition}", f"{DECK['completed'][condition]} / {DECK['episodes']}",
         f"{stats.get('completed')} / {stats.get('episodes')}" if stats else "—"),
        (f"Perfect documents, {condition}", f"{DECK['perfect_documents'][condition]} / {DECK['episodes']}",
         f"{stats.get('perfect_documents')} / {stats.get('episodes')}" if stats else "—"),
        (f"Median input tokens, {condition}", cell(DECK["median_input_tokens"][condition], "int"),
         cell(stats["input_tokens"]["median"] if stats else None, "int")),
        (f"Mean input tokens, {condition}", cell(DECK["mean_input_tokens"][condition], "k"),
         cell(stats["input_tokens"]["mean"] if stats else None, "k")),
        (f"Top-5 document input share, {condition}", f"{DECK['top5_input_share_pct'][condition]}%",
         cell((mcrdr["top_document_input_share"].get(condition) or {}).get("share", 0) * 100
              if condition in mcrdr["top_document_input_share"] else None, "pct")),
    ]
for run in ("Run 1", "Run 2", "Run 3", "Pooled"):
    differences = (mcrdr["pooled"] if run == "Pooled" else mcrdr["runs"].get(run, {})).get("differences", {})
    value = differences.get("reduced_minus_full")
    rows.append((f"Reduced − full, {run}", cell(DECK["reduced_minus_full_pp"][run], "pp"),
                 cell(None if value is None else value * 100, "pp")))

lines = ["| Metric | Deck (SCRDR) | MCRDR |", "|---|---|---|"] + [f"| {a} | {b} | {c} |" for a, b, c in rows]
display(Markdown("\n".join(lines)))

## 10. Download the results

In [ ]:
import shutil

ARCHIVE = Path("/content/mcrdr-results.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPERIMENTS_DIR.rglob("*")):
        relative = path.relative_to(EXPERIMENTS_DIR)
        keep = (relative.parts[0] in ("results", "report", "configs", "logs")
                or relative.name in ("report-spec.json", "pdf-snapshot-mcrdr.json"))
        if path.is_file() and keep:
            archive.write(path, relative.as_posix())
print(f"Wrote {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except Exception:
    print("Automatic download is unavailable here; fetch the file from the runtime's file browser.")